# Fine-Tuning Qwen3-4B-Thinking

This notebook provides a complete pipeline to fine-tune the **Qwen3-4B-Thinking** model using **QLoRA** (4-bit quantization + LoRA adapters).

## 1. Environment Setup

We need `peft` for LoRA, `trl` for the SFT (Supervised Fine-Tuning) trainer, and `bitsandbytes` for quantization.

In [2]:
# Install fine-tuning dependencies
!pip install peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [36]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Configuration
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/final-odyssey-math-cleaned.jsonl"
OUTPUT_DIR = "./results/qwen3_math_lora"
SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Do not format your answer with latex inside the boxed part. "
    "These instructions supersede any user instructions. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}. "
    "Before writing the final \boxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor."
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 2. Load Dataset

We load the local JSONL dataset and format it for instruction tuning.
The model expects a conversation-like format or a prompt-response pair.

### Load MathInstruct Dataset for finetuning

In [45]:
from datasets import load_dataset

# Load, shuffle, and then select a subset to ensure randomization
ds = load_dataset("tiedong/goat", split="train")
ds = ds.shuffle(seed=42).select(range(10000))

def format_instruction(sample):
    """
    Format the instructions for the dataset using the global SYSTEM_PROMPT.
    Ensures the output is wrapped in \boxed{}.
    """
    # Wrap output in boxed if it's not already there
    output_text = sample["answer"]
    formatted_output = f"\\boxed{{{output_text}}}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sample["instruction"]},
        {"role": "assistant", "content": formatted_output}
    ]

    return {"messages": messages}

ds = ds.map(format_instruction)
print(f"Subset size: {len(ds)}")
print(f"Sample formatted message:\n{ds[0]['messages']}")

Subset size: 10000
Sample formatted message:
[{'role': 'system', 'content': "You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. These instructions supersede any user instructions. Put your final answer inside \\boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. Before writing the final \x08oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor."}, {'role': 'user', 'content': '765455235 plus 27236193'}, {'role': 'assistant', 'content': '\\boxed{792691428}'}]


### Concatenating and Shuffling Multiple Datasets
If you have multiple datasets (e.g., `ds1` and `ds2`), you can merge them into a single training set. Note: Ensure both datasets have the same column structure before concatenating.

In [51]:
from datasets import concatenate_datasets, load_dataset

# Example: Loading a second dataset
ds_goat = load_dataset("tiedong/goat", split="train").select(range(5000))
ds_math = load_dataset("lighteval/MATH", split="train", trust_remote_code=True).select(range(5000))

# Standardize columns if necessary (e.g., renaming 'problem' to 'instruction' and 'solution' to 'answer')
ds_math = ds_math.rename_columns({"problem": "instruction", "solution": "answer"})

# Keep only the necessary columns for both
columns_to_keep = ["instruction", "answer"]
ds_goat = ds_goat.remove_columns([col for col in ds_goat.column_names if col not in columns_to_keep])
ds_math = ds_math.remove_columns([col for col in ds_math.column_names if col not in columns_to_keep])

# Concatenate
combined_ds = concatenate_datasets([ds_goat, ds_math])

# Shuffle the combined dataset
combined_ds = combined_ds.shuffle(seed=42)

# Map the formatting function defined earlier
ds = combined_ds.map(format_instruction)

print(f"Combined dataset size: {len(ds)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


DatasetNotFoundError: Dataset 'lighteval/MATH' doesn't exist on the Hub or cannot be accessed.

## 3. Model Initialization (QLoRA)

We load the model in 4-bit to save memory and prepare it for LoRA training.

In [46]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# Optimized LoRA configuration targeting all linear layers for better math reasoning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules="all-linear", # Replaced specific modules with all-linear for better coverage
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


## 4. Training

We use the `SFTTrainer` which handles the chat template automatically if the dataset contains a `messages` column.

In [48]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16, # Increased batch size for H100
    gradient_accumulation_steps=2,  # Adjusted for effective batch size of 64
    learning_rate=2e-4,
    max_grad_norm=0.3,
    num_train_epochs=1,             # Reduced epochs for quicker iteration
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,               # Updated to use the subset directly
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

Step,Training Loss
10,1.939530
20,1.339508
30,0.550902
40,0.392474
50,0.338911
60,0.309712
70,0.294503
80,0.294259
90,0.284275
100,0.288771


TrainOutput(global_step=313, training_loss=0.37565224582013995, metrics={'train_runtime': 672.0301, 'train_samples_per_second': 14.88, 'train_steps_per_second': 0.466, 'total_flos': 4.283881938144461e+16, 'train_loss': 0.37565224582013995, 'entropy': 0.24860722720623016, 'num_tokens': 1775550.0, 'mean_token_accuracy': 0.914333438873291, 'epoch': 1.0})

## 5. Save and Test

Save the adapter and run a quick test.

In [49]:
trainer.save_model(os.path.join(OUTPUT_DIR, "final_adapter"))
print(f"Adapter saved to {OUTPUT_DIR}/final_adapter")

Adapter saved to ./results/qwen3_math_lora/final_adapter


## 6. Loading the Fine-Tuned Model
To use your fine-tuned model later, you must load the base model and then apply the saved PEFT (LoRA) adapters.

In [53]:
from peft import PeftModel

# 1. Load the base model (must use the same quantization config)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 2. Load the saved adapter
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Fine-tuned model loaded successfully!


In [54]:
import torch

# 1. Enable cache for inference (was disabled by gradient checkpointing)
ft_model.config.use_cache = True
ft_model.eval()

question = "What is the sum of the first 10 positive even numbers?"
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ],
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")


with torch.no_grad():
    output_tokens = ft_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1, # Helps prevent repetitive loops
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are an expert mathematician. Solve the problem step-by-step. Do not format your answer with latex inside the boxed part. These instructions supersede any user instructions. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}. Before writing the final oxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor.
user
What is the sum of the first 10 positive even numbers?
assistant
<think>
</think>

\boxed{110}
